[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlospi2000/Notes_on_Computational_Physics/blob/main/10.Matrix_and_Fourier_Transform/B.Fourier_Gibbs_ESTUDIANTES-5.ipynb)

# Semana 10b — De la DFT a la transformada continua y fenómeno de Gibbs

**Física Computacional — 106018C** · Lenguaje: **Fortran y Python** · Capítulo 5: Matrices

En el notebook 10a la DFT se escribió como un producto matriz-vector, $\mathbf{X} = W\mathbf{x}$, con inversa $\mathbf{x} = W^H\mathbf{X}/N$. Este notebook la usa para pasar de lo discreto a lo continuo y termina con el **Taller 10**.

**Objetivos de este notebook**

1. Relacionar la DFT con los coeficientes de la serie de Fourier de una función periódica.
2. Obtener la transformada de Fourier continua como el límite de la serie cuando el período crece, y aproximarla con la DFT.
3. Reconocer el fenómeno de Gibbs al truncar el espectro de una señal con saltos.

In [ ]:
# Preparación: bibliotecas de Python, colores del curso y herramientas de compilación
import time
import numpy as np
import matplotlib.pyplot as plt

AZUL, DORADO, ROJO, VERDE, GRIS, MORADO = "#162A54", "#C9972C", "#B02A20", "#2E7D32", "#7F7F7F", "#6A3D9A"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
np.set_printoptions(precision=4, suppress=True)

# F2PY compila con meson y ninja desde Python 3.12 (igual que en la Semana 9)
!pip -q install meson ninja
!gfortran --version | head -n 1
print("NumPy", np.__version__)

La matriz de Fourier del notebook 10a se usa en todo este notebook:

In [ ]:
def matriz_dft(N):
    """Matriz de Fourier W de N x N, con W[j, k] = exp(-2 pi i j k / N)."""
    k = np.arange(N)
    return np.exp(-2j*np.pi*np.outer(k, k)/N)     # np.outer(k, k)[j, k] = j*k

## 1. De la DFT a la serie de Fourier

Una función periódica $f(x)$ de período $L$ se puede escribir como una **serie de Fourier**,

$$f(x) = \frac{a_0}{2} + \sum_{n=1}^{\infty}\left[a_n\cos\left(\frac{2\pi n x}{L}\right) + b_n\sin\left(\frac{2\pi n x}{L}\right)\right],$$

con los coeficientes

$$a_n = \frac{2}{L}\int_0^L f(x)\cos\left(\frac{2\pi n x}{L}\right)dx, \qquad b_n = \frac{2}{L}\int_0^L f(x)\sin\left(\frac{2\pi n x}{L}\right)dx,$$

donde $a_0/2$ es el promedio de $f$ en un período, y $a_n$, $b_n$ son las amplitudes de los cosenos y senos de $n$ ciclos por período. Es la descomposición de la figura inicial del notebook 10a, ahora para una función continua: las integrales son los productos escalares de la sección 1.1 de 10a, con integrales en lugar de sumas.

Si se muestrea un período en $N$ puntos, $x_k = kL/N$ con $k = 0, \dots, N-1$, y se aproxima cada integral con la **regla del trapecio** (Semana 7), los extremos se juntan porque $f(0) = f(L)$ y queda

$$a_n \approx \frac{2}{N}\sum_{k=0}^{N-1} f(x_k)\cos\left(\frac{2\pi nk}{N}\right) = \frac{2}{N}\,\mathrm{Re}\,X_n, \qquad
b_n \approx \frac{2}{N}\sum_{k=0}^{N-1} f(x_k)\sin\left(\frac{2\pi nk}{N}\right) = -\frac{2}{N}\,\mathrm{Im}\,X_n .$$

**La DFT de las muestras de un período es la regla del trapecio aplicada a las integrales de los coeficientes.** Para funciones periódicas y suaves, el trapecio converge mucho más rápido que $h^2$, por lo que unas decenas de muestras bastan.

### 1.1 Onda cuadrada

La onda cuadrada de período $2\pi$, $f(x) = 1$ en $0 < x < \pi$ y $f(x) = -1$ en $\pi < x < 2\pi$, tiene la serie

$$f(x) = \frac{4}{\pi}\left(\sin x + \frac{\sin 3x}{3} + \frac{\sin 5x}{5} + \cdots\right), \qquad b_n = \frac{4}{n\pi}\ (n \text{ impar}), \quad a_n = 0 .$$

En los saltos ($x = 0$ y $x = \pi$) se toma el valor medio, $f = 0$, que es el valor al que converge la serie en una discontinuidad. La figura muestra la **síntesis**: la suma parcial con los términos hasta $n = M$ se acerca a la onda cuadrada a medida que se agregan términos.

In [ ]:
# Figura: síntesis de la onda cuadrada con los términos impares hasta n = M
xx = np.linspace(0, 4*np.pi, 1000)
cuadrada = np.sign(np.sin(xx))

fig, ax = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for a_, M, color in zip(ax, [1, 5, 25], [AZUL, VERDE, ROJO]):
    S = sum(4/(n*np.pi)*np.sin(n*xx) for n in range(1, M + 1, 2))
    a_.plot(xx, cuadrada, color=GRIS, lw=1)
    a_.plot(xx, S, color=color, lw=1.5)
    a_.set(title=f"términos hasta n = {M}", xlabel="x", xticks=[0, np.pi, 2*np.pi, 3*np.pi, 4*np.pi],
           xticklabels=["0", "π", "2π", "3π", "4π"])
ax[0].set_ylabel("f(x)")
plt.tight_layout(); plt.show()

In [ ]:
# Coeficientes de la onda cuadrada a partir de la DFT de N muestras de un período
N  = 256
xk = 2*np.pi*np.arange(N)/N
fk = np.where(np.arange(N) < N//2, 1.0, -1.0)   # +1 en (0, pi), -1 en (pi, 2 pi)
fk[0] = fk[N//2] = 0.0                # valor medio en los saltos x = 0 y x = pi
Xn = matriz_dft(N) @ fk
a  = 2*Xn.real/N
b  = -2*Xn.imag/N

print(" n     b_n (DFT)    4/(n pi)     error")
for n in range(1, 12, 2):
    print(f"{n:2d}   {b[n]:10.6f}   {4/(n*np.pi):10.6f}   {abs(b[n] - 4/(n*np.pi)):.1e}")
print("máx |a_n| =", np.max(np.abs(a[:N//2])), "  |b_2| =", abs(b[2]))

# Figura: coeficientes b_n (dominio de la frecuencia)
n = np.arange(1, 40)
fig, ax = plt.subplots(figsize=(7, 3))
ax.stem(n, b[1:40], linefmt=DORADO, markerfmt="o", basefmt=GRIS, label="DFT")
ax.plot(n[::2], 4/(n[::2]*np.pi), "--", color=AZUL, label="4/(nπ)")
ax.set(xlabel="n", ylabel="b_n", title="Coeficientes de la onda cuadrada"); ax.legend()
plt.tight_layout(); plt.show()

Los $b_n$ de índice par y todos los $a_n$ salen nulos, y los $b_n$ impares coinciden con $4/(n\pi)$ con un error que crece con $n$: para $n$ cercano a $N/2$ el muestreo ya no alcanza, que es el aliasing del notebook 10a. Los coeficientes de la onda cuadrada decaen solo como $1/n$, lentamente, porque la función tiene saltos. Esa lentitud es la causa del fenómeno de Gibbs de la sección 4.

## 2. De la serie a la transformada de Fourier continua

Una función que no es periódica se puede pensar como periódica con período $L \to \infty$. La figura siguiente lo muestra con un pulso rectangular de ancho 1 que se repite con período $L$. La serie de Fourier tiene líneas en las frecuencias $n/L$, separadas por $1/L$. Al duplicar $L$, las líneas se juntan al doble de densidad, pero siguen la **misma curva envolvente**. En el límite $L \to \infty$ las líneas llenan el eje y la envolvente es la transformada continua del pulso aislado, la función $\mathrm{sinc}(f) = \sin(\pi f)/(\pi f)$.

In [ ]:
# Figura: al crecer el período L, las líneas del espectro se juntan y llenan la transformada continua
dt = 1/32                                       # paso de muestreo [s]
fc = np.linspace(-4, 4, 800)                    # frecuencia "continua" [Hz]

fig, ax = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for a_, L in zip(ax, [2, 4, 8]):
    N = int(L/dt)
    tt = -L/2 + np.arange(N)*dt                 # un período, centrado en t = 0
    pulso = np.where(np.abs(tt) < 0.5, 1.0, 0.0)
    pulso[np.isclose(np.abs(tt), 0.5)] = 0.5    # valor medio en los bordes
    j = np.arange(N)
    fj = np.where(j <= N//2, j, j - N)/L        # frecuencias n/L [Hz]
    Xc = dt*np.exp(-2j*np.pi*fj*tt[0])*(matriz_dft(N) @ pulso)   # L c_n (sección 2.1)
    sel = np.abs(fj) <= 4
    a_.plot(fc, np.sinc(fc), color=GRIS, lw=1, label="sinc(f), pulso aislado")
    a_.stem(fj[sel], Xc.real[sel], linefmt=DORADO, markerfmt="o", basefmt=" ")
    a_.set(ylabel="L c_n", title=f"período L = {L} s: líneas separadas 1/L = {1/L:g} Hz")
ax[0].legend(loc="upper right", fontsize=8); ax[-1].set_xlabel("f [Hz]")
plt.tight_layout(); plt.show()

La suma de la serie se convierte en una integral y resulta la **transformada de Fourier**,

$$\hat{x}(f) = \int_{-\infty}^{\infty} x(t)\, e^{-2\pi i f t}\, dt, \qquad x(t) = \int_{-\infty}^{\infty} \hat{x}(f)\, e^{+2\pi i f t}\, df,$$

donde $t$ es el tiempo [s], $f$ la frecuencia [Hz] y $\hat{x}(f)$ la amplitud compleja por unidad de frecuencia. Con la frecuencia angular $\omega = 2\pi f$ se escribe $\hat{x}(\omega) = \int x(t)e^{-i\omega t}dt$, y la inversa lleva el factor $1/2\pi$; es la misma transformada con otra variable.

### 2.1 La DFT como aproximación de la transformada continua

Si $x(t)$ es prácticamente nula fuera de un intervalo $[t_0,\ t_0 + N\Delta t)$, la integral se aproxima con la suma de rectángulos sobre las muestras $x_k = x(t_0 + k\Delta t)$. En las frecuencias $f_j = j/(N\Delta t)$:

$$\hat{x}(f_j) \approx \Delta t\sum_{k=0}^{N-1} x_k\, e^{-2\pi i f_j (t_0 + k\Delta t)} = \Delta t\; e^{-2\pi i f_j t_0}\; X_j ,$$

donde

- $\Delta t$ convierte la suma en una aproximación de la integral;
- $e^{-2\pi i f_j t_0}$ es un **factor de fase** que corrige que las muestras empiecen en $t_0$ y no en 0;
- $X_j$ es la DFT de las muestras.

Para los coeficientes con $j > N/2$ se usa la frecuencia negativa $f_j = (j - N)/(N\Delta t)$, por la simetría del notebook 10a. Es la fórmula que se usó en la figura anterior.

### 2.2 Verificación con una gaussiana

La función $x(t) = e^{-\pi t^2}$ es su propia transformada: $\hat{x}(f) = e^{-\pi f^2}$. Sirve como caso de prueba con resultado conocido.

In [ ]:
# Transformada continua de exp(-pi t^2) aproximada con la DFT
N, dt, t0 = 256, 1/16, -8.0          # muestras en [-8, 8) s
t = t0 + np.arange(N)*dt
x = np.exp(-np.pi*t**2)

j  = np.arange(N)
fj = np.where(j <= N//2, j, j - N)/(N*dt)        # frecuencias positivas y negativas [Hz]
X_cont = dt*np.exp(-2j*np.pi*fj*t0)*(matriz_dft(N) @ x)

orden = np.argsort(fj)                           # ordena de f negativa a positiva
print("máx |error| frente a exp(-pi f^2) =", np.max(np.abs(X_cont - np.exp(-np.pi*fj**2))))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(t, x, color=AZUL)
ax[0].set(xlabel="t [s]", ylabel="x(t)", xlim=(-3, 3), title="Dominio del tiempo")
ax[1].plot(fj[orden], X_cont.real[orden], "o", color=DORADO, ms=3, label="DFT")
ax[1].plot(fj[orden], np.exp(-np.pi*fj[orden]**2), color=AZUL, label="exacta")
ax[1].set(xlabel="f [Hz]", ylabel="Re x̂(f)", xlim=(-3, 3), title="Dominio de la frecuencia")
ax[1].legend(); plt.tight_layout(); plt.show()

El error es del orden del redondeo porque la gaussiana y su transformada decaen muy rápido: el intervalo $[-8, 8)$ contiene toda la función y la frecuencia de Nyquist, $8$ Hz, toda la transformada. En general la DFT aproxima la transformada continua con dos errores, que son los del notebook 10a: el **truncamiento** del intervalo de tiempo (leakage) y el **muestreo** (aliasing).

## 3. Resumen: de lo discreto a lo continuo

| | Señal | Frecuencias | Herramienta |
|---|---|---|---|
| DFT | $N$ muestras | $N$ valores discretos $f_j = j/T$ | $\mathbf{X} = W\mathbf{x}$ |
| Serie de Fourier | función periódica de período $L$ | infinitas, discretas $n/L$ | integrales de un período |
| Transformada continua | función no periódica | continuas | integral en todo $t$ |

La DFT es la única de las tres que se calcula exactamente con un computador; las otras dos se aproximan con ella.

## 4. Truncar el espectro: el fenómeno de Gibbs

Con los coeficientes de Fourier se puede reconstruir la señal usando solo las frecuencias bajas: se conservan los coeficientes con $|f_j| \le K\Delta f$, se anulan los demás y se aplica la transformada inversa. Es un **filtro pasa-bajos ideal**. Como las frecuencias bajas están en los dos extremos del arreglo (figura de la sección 5.2 de 10a), se conservan los índices

$$j = 0, 1, \dots, K \qquad \text{y} \qquad j = N-K, \dots, N-1 .$$

Para una señal suave, pocas frecuencias bastan. Para una señal con saltos aparecen oscilaciones junto a cada discontinuidad, y la mayor de ellas **no disminuye** al aumentar $K$: se hace más angosta y se acerca al salto, pero su altura se mantiene cerca del 9 % del salto. Ya se ve en la síntesis de la onda cuadrada de la sección 1. Es el **fenómeno de Gibbs**. Para la serie de Fourier de una función continua, la altura límite es

$$\frac{1}{2} + \frac{1}{\pi}\,\mathrm{Si}(\pi) \approx 1.0895 \quad \text{para un salto de 0 a 1}, \qquad \mathrm{Si}(z) = \int_0^z \frac{\sin u}{u}\,du ,$$

es decir, un sobrepaso del 8.95 % del salto. El Taller 10 lo mide con su propia librería.

In [ ]:
# Figura: función caja de 128 muestras (39 ceros, 50 unos, 39 ceros) y reconstrucciones
N = 128
caja = np.zeros(N); caja[39:89] = 1.0
W128 = matriz_dft(N)
Xc = W128 @ caja

j = np.arange(N)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.4]})
ax[0].semilogy(j, np.abs(Xc) + 1e-16, "o", color=AZUL, ms=3)
ax[0].axvspan(-0.5, 16.5, color=VERDE, alpha=0.15); ax[0].axvspan(N - 16.5, N - 0.5, color=VERDE, alpha=0.15)
ax[0].set(xlabel="índice j", ylabel="|X_j|", title="Espectro: se conserva lo sombreado (K = 16)", ylim=(1e-1, 1e2))
ax[1].plot(j, caja, "k-", drawstyle="steps-mid", lw=1, label="caja")
for K, color in [(4, ROJO), (16, DORADO), (32, AZUL)]:
    conservar = (j <= K) | (j >= N - K)                  # frecuencias |j| <= K
    rec = (W128.conj().T @ np.where(conservar, Xc, 0)) / N
    ax[1].plot(j, rec.real, color=color, label=f"K = {K}")
ax[1].set(xlabel="k", ylabel="x_k", title="Reconstrucción con las frecuencias |j| ≤ K")
ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**No anule solo un extremo del arreglo de coeficientes.** Si se anulan únicamente los últimos $n$ coeficientes, $X_{N-n}, \dots, X_{N-1}$, se eliminan las frecuencias negativas pero no sus parejas positivas, se rompe la simetría $X_{N-j} = \overline{X_j}$, y la señal reconstruida sale **compleja**: con la caja anterior y $n = 20$, la parte imaginaria llega a 0.67. Las frecuencias altas de una señal real están en el **centro** del arreglo, alrededor de $j = N/2$.

</div>

<div style="border: 2px solid #C9972C; background-color: #FBF6EA; padding: 10px 14px; margin: 8px 0;">

**Ejercicio de calentamiento 3 (no calificado): Gibbs en la serie de la onda cuadrada**

La suma parcial de la onda cuadrada de la sección 1.1, con los términos impares hasta $n = M$, es

$$S_M(x) = \frac{4}{\pi}\sum_{\substack{n = 1 \\ n\ \text{impar}}}^{M} \frac{\sin(nx)}{n}.$$

Junto al salto en $x = 0$, donde la onda pasa de $-1$ a $1$, la suma parcial sobrepasa el valor 1. En la celda siguiente, para $M = 9, 49$ y $199$:

1. Obtenga $S_M(x)$ en la malla dada, que cubre la zona cercana al salto.
2. Obtenga su valor máximo, el sobrepaso como porcentaje del salto, $(S_{\max} - 1)/2 \times 100\,\%$, y la posición $x_{\max}$ del máximo.
3. Calcule $x_{\max}(M+1)/\pi$.

**Debe obtener:** $S_{\max} = 1.1823,\ 1.1791,\ 1.1790$; sobrepasos de $9.12\,\%$, $8.96\,\%$ y $8.95\,\%$, que se acercan al $8.95\,\%$ de la sección 4 en lugar de desaparecer; y $x_{\max}(M+1)/\pi = 1.000$: el máximo está en $x_{\max} = \pi/(M+1)$ y se acerca al salto a medida que $M$ crece.

</div>

In [ ]:
# Ejercicio de calentamiento 3
for M in [9, 49, 199]:
    x = np.linspace(1e-6, 6*np.pi/(M + 1), 20001)    # zona cercana al salto en x = 0
    # TODO: obtenga S_M(x), su máximo, el sobrepaso y x_max (M+1)/pi

## 5. Taller 10: librería de Fourier en Fortran y fenómeno de Gibbs

**Entrega:** viernes 4 de diciembre de 2026.

La DFT matricial del notebook 10a se convierte en una librería de Fortran, que se usa desde Python para estudiar el fenómeno de Gibbs. En la Semana 11 se presentará la FFT, que obtiene los mismos coeficientes mucho más rápido.

### A. Enunciado y requisitos

**Requisitos**

1. Las celdas del taller trabajan dentro de la carpeta `taller_fourier`, que crea la celda de la Parte B. En el repositorio, la carpeta `taller_fourier` contiene `modulo_fourier.f90` y este notebook.
2. El módulo se llama `fourier`, está en `modulo_fourier.f90` y se compila como la librería `fou`:
   `python -m numpy.f2py -c modulo_fourier.f90 -m fou`
3. El módulo contiene tres subrutinas que la celda de verificación llama directamente:

| Subrutina | Entradas | Salida | Uso desde Python |
|---|---|---|---|
| `matriz_dft` | `n` | `w(n,n)`, complejo: $W_{jk} = e^{-2\pi i\,(j-1)(k-1)/n}$ | `fou.fourier.matriz_dft(n)` |
| `dft` | `n`; `x(n)`, complejo | `xf(n)` $= W\,x$ | `fou.fourier.dft(x)` |
| `idft` | `n`; `xf(n)`, complejo | `x(n)` $= W^H\,xf/n$ | `fou.fourier.idft(xf)` |

   Los índices de Fortran empiezan en 1: el elemento `w(j,k)` corresponde a $W_{j-1,\,k-1}$ del notebook 10a.
4. `dft` e `idft` obtienen $W$ llamando a `matriz_dft` y calculan el producto con `matmul`. La inversa usa la transpuesta conjugada, no un sistema de ecuaciones.
5. Se usan `complex(8)` y `real(8)`. La matriz $W$ dentro de `dft` e `idft` debe ser **dinámica** (`allocatable`, Capítulo 3): con $n = 1024$ ocupa 16 MiB, y como arreglo automático puede desbordar la memoria de pila. El módulo no usa `stop`.

**Qué se entrega**

- En la carpeta del repositorio: `modulo_fourier.f90` y este notebook, con la celda de verificación en 3 de 3 PASS y las respuestas de la sección D en celdas Markdown al final.
- En Moodle: el reporte de una página y el enlace a la carpeta.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**No está permitido** usar `np.fft`, `scipy.fft` ni la función `matriz_dft` de Python de este notebook dentro de las actividades. Las transformadas se calculan con su librería de Fortran. `np.fft.fft` solo puede usarse para comparar, en la actividad 1.

</div>

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Precisión del exponente.** El ángulo $2\pi (j-1)(k-1)/n$ crece hasta valores del orden de $2\pi n$. Como $e^{-2\pi i\,m/n}$ solo depende del residuo de $m$ al dividir por $n$, conviene usar `mod((j-1)*(k-1), n)` en lugar de $(j-1)(k-1)$: el ángulo queda en $[0, 2\pi)$ y el error de redondeo no crece con $n$.

</div>

### B. Parte Fortran: la librería

Ejecute primero la celda siguiente, que crea la carpeta `taller_fourier` y la deja como carpeta de trabajo. Después escriba `modulo_fourier.f90` y compílelo. Recuerde de la Semana 9: después de cada recompilación, **reinicie el entorno** y ejecute de nuevo la celda de preparación y esta celda.

In [ ]:
# Carpeta de trabajo del taller: aquí se escriben, compilan, importan y verifican los archivos
import os, sys
CARPETA = "taller_fourier"
if os.path.basename(os.getcwd()) != CARPETA:     # evita crear taller_fourier/taller_fourier
    os.makedirs(CARPETA, exist_ok=True)
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())               # import fou busca la librería aquí
print("Carpeta de trabajo:", os.getcwd())

In [ ]:
# Compila la librería "fou" (ejecute esta celda cuando su archivo esté listo)
!python -m numpy.f2py -c modulo_fourier.f90 -m fou 2>&1 | tail -n 3

### C. Parte Python: uso de la librería

Ejecute las actividades en las celdas siguientes. Cada una indica qué debe obtener.

**Actividad 1. Prueba de la librería.** Importe `fou` e imprima la documentación de `dft`. Con su librería, obtenga:

- la matriz `matriz_dft(4)`, que debe coincidir con la del notebook 10a;
- la DFT de $x = (1, 2, 3, 4)$, que debe ser $(10,\ -2+2i,\ -2,\ -2-2i)$;
- para un vector aleatorio de 64 elementos, el error máximo de `dft` frente a `np.fft.fft`, el de `idft(dft(x))` frente a $x$, y el de $W W^H$ frente a $64\,I$. Los tres deben ser menores que $10^{-10}$.

**Actividad 2. Espectro de la función caja.** Construya la caja de $N = 1024$ muestras: 312 ceros, 400 unos y 312 ceros. Es la misma proporción de la caja de la sección 4, con más resolución. Su DFT tiene forma analítica:

$$|X_0| = L, \qquad |X_j| = \left|\frac{\sin(\pi j L/N)}{\sin(\pi j/N)}\right| \quad (j \neq 0),$$

con $L = 400$, el número de unos. Obtenga la gráfica de $|X_j|$ para $j = 0, \dots, N/2$, en escala logarítmica, junto con la fórmula, y el error máximo entre las dos, que debe ser menor que $10^{-8}$.

**Actividad 3. Filtro pasa-bajos.** Para $K = 8, 16, 32, 64$, conserve los coeficientes con $|j| \le K$ (sección 4), anule los demás y reconstruya la señal con `idft`. Obtenga:

- la parte imaginaria máxima de cada reconstrucción, que debe ser menor que $10^{-10}$;
- una figura con la caja y las cuatro reconstrucciones, ampliada alrededor del primer salto ($250 \le k \le 400$).

**Actividad 4. Medición del fenómeno de Gibbs.** El primer salto de la caja está entre las muestras $k = 311$ y $k = 312$ (numeración de Python), es decir, en $k_s = 311.5$. Para cada $K$ de la actividad 3, obtenga una tabla con:

- el valor máximo $s_{\max}$ de la reconstrucción y el sobrepaso $(s_{\max} - 1)\times 100\,\%$ (el salto es de 0 a 1);
- la distancia $d$ entre la posición del máximo y el salto más cercano, y el producto $K\,d$;
- el error cuadrático medio $E_K = \sqrt{\frac{1}{N}\sum_k (s_k - x_k)^2}$ y la razón $E_K/E_{2K}$.

Debe obtener sobrepasos entre 7.5 % y 9 % para todos los $K$, productos $K\,d$ cercanos a $N/2 = 512$ y razones $E_K/E_{2K}$ cercanas a $\sqrt{2} \approx 1.41$.

In [ ]:
# Actividad 1

In [ ]:
# Actividad 2

In [ ]:
# Actividad 3

In [ ]:
# Actividad 4

### D. Preguntas de interpretación

Responda en celdas Markdown al final de este notebook, con los números de sus resultados:

1. **Simetría.** ¿Por qué la reconstrucción de la actividad 3 sale real? ¿Qué pasaría si se anularan solo los últimos coeficientes del arreglo, como advierte la sección 4?
2. **Gibbs.** Según su tabla de la actividad 4, ¿el sobrepaso disminuye al aumentar $K$? ¿Qué cantidades sí cambian, y cómo? Compare su sobrepaso con el 8.95 % de la serie de Fourier.
3. **Costo.** ¿Cuántas multiplicaciones complejas hace su subrutina `dft` con $N = 1024$? ¿Por qué factor se multiplicaría el tiempo si $N$ se duplicara?

### E. Verificación

La celda siguiente **no debe modificarse**. Importa su librería y la prueba con casos de resultado conocido. Si recompiló la librería, reinicie el entorno antes de ejecutarla. El taller exige los tres PASS.

In [ ]:
# ==================== VERIFICACIÓN DEL TALLER 10 (no modificar) ====================
import os, sys, glob
import numpy as np

resultados = []
def caso(nombre, funcion):
    try:
        detalle = funcion()
        ok = True
    except Exception as e:
        ok, detalle = False, str(e)
    resultados.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f" -> {detalle}" if detalle else ""))

# La verificación siempre trabaja dentro de la carpeta del taller
CARPETA = "taller_fourier"
if os.path.basename(os.getcwd()) != CARPETA:
    if not os.path.isdir(CARPETA):
        raise SystemExit(f"No existe la carpeta {CARPETA}: ejecute la celda de la Parte B.")
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Verificando en:", os.getcwd(), "\n")

def libreria():
    if not os.path.isfile("modulo_fourier.f90"):
        raise ValueError(f"no existe modulo_fourier.f90 en {CARPETA}")
    so = glob.glob("fou.*.so") + glob.glob("fou.*.pyd")
    if not so:
        raise ValueError(f"no se encontró la librería fou en {CARPETA}; compile con F2PY")
    so = max(so, key=os.path.getmtime)                # la más reciente, si hay varias
    if os.path.getmtime(so) < os.path.getmtime("modulo_fourier.f90"):
        raise ValueError("la librería es anterior a modulo_fourier.f90; recompile y reinicie")
    import fou
    if os.path.abspath(fou.__file__) != os.path.abspath(so):
        raise ValueError(f"la librería cargada es {fou.__file__}, no la de {CARPETA}; reinicie el entorno")
    if not hasattr(fou, "fourier"):
        raise ValueError("la librería no contiene el módulo fourier")
    return fou.fourier

def dft_directa(x):
    """DFT por la definición, como suma doble (referencia de la verificación)."""
    N = len(x); k = np.arange(N)
    return np.array([np.sum(x*np.exp(-2j*np.pi*j*k/N)) for j in range(N)])

def c1():
    lib = libreria()
    W4 = np.array([[1, 1, 1, 1], [1, -1j, -1, 1j], [1, -1, 1, -1], [1, 1j, -1, -1j]])
    if np.max(np.abs(lib.matriz_dft(4) - W4)) > 1e-12:
        raise ValueError("matriz_dft(4) no es la matriz de Fourier de 4 x 4")
    X = lib.dft([1.0, 2.0, 3.0, 4.0])
    if np.max(np.abs(X - np.array([10, -2 + 2j, -2, -2 - 2j]))) > 1e-12:
        raise ValueError("dft([1, 2, 3, 4]) no da (10, -2+2i, -2, -2-2i)")
    return "matriz de 4 x 4 y DFT de (1, 2, 3, 4) correctas"

def c2():
    lib = libreria()
    x = np.random.default_rng(11).normal(size=64) + 1j*np.random.default_rng(12).normal(size=64)
    if np.max(np.abs(lib.dft(x) - dft_directa(x))) > 1e-10:
        raise ValueError("dft no coincide con la definición para un vector complejo de 64 elementos")
    if np.max(np.abs(lib.idft(lib.dft(x)) - x)) > 1e-12:
        raise ValueError("idft(dft(x)) no recupera x")
    W = lib.matriz_dft(64)
    if np.max(np.abs(W @ W.conj().T - 64*np.eye(64))) > 1e-10:
        raise ValueError("W W^H no es 64 I")
    return "dft, idft y ortogonalidad correctas para N = 64"

def c3():
    lib = libreria()
    N, L, K = 1024, 400, 64
    caja = np.zeros(N); caja[312:712] = 1.0
    X = lib.dft(caja)
    j = np.arange(1, N//2 + 1)
    if np.max(np.abs(np.abs(X[1:N//2 + 1]) - np.abs(np.sin(np.pi*j*L/N)/np.sin(np.pi*j/N)))) > 1e-8:
        raise ValueError("el espectro de la caja no coincide con la fórmula analítica")
    k = np.arange(N)
    s = lib.idft(np.where((k <= K) | (k >= N - K), X, 0))
    if np.max(np.abs(s.imag)) > 1e-10:
        raise ValueError("la reconstrucción filtrada no es real")
    smax = s.real.max()
    d = np.min(np.abs(np.argmax(s.real) - np.array([311.5, 711.5])))
    if not (1.085 < smax < 1.093) or not (400 < K*d < 600):
        raise ValueError(f"Gibbs con K = 64 fuera de lo esperado: s_max = {smax:.4f}, K d = {K*d:.0f}")
    return f"espectro de la caja correcto; Gibbs con K = 64: s_max = {smax:.4f}, K d = {K*d:.0f}"

caso("1. Matriz de Fourier y DFT de 4 elementos", c1)
caso("2. DFT, inversa y ortogonalidad con N = 64", c2)
caso("3. Espectro de la caja y fenómeno de Gibbs", c3)
print(f"\nResultado: {sum(resultados)} de {len(resultados)} PASS")